In [14]:
import shutil
import tarfile
from pathlib import Path

archive = Path("../data/processed/pylate_index.tar.gz")
index_dir = Path("../data/processed/pylate_index")

if not index_dir.exists():
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(path="../data/processed")

In [15]:
import subprocess, time, requests

try:
    requests.get("http://localhost:11434", timeout=2)
    print("Ollama already running")
except Exception:
    subprocess.Popen(["ollama", "serve"])
    time.sleep(5)
    print("Ollama started")

subprocess.run(["ollama", "pull", "llama3.2"], check=True)


Ollama already running


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


CompletedProcess(args=['ollama', 'pull', 'llama3.2'], returncode=0)

In [16]:
# Initialize the colBERT retriever

from pylate import retrieve
from pylate import indexes, models
index = indexes.Voyager(
    index_folder="../data/processed/pylate_index",
    index_name="esci_data_index",
)

retriever = retrieve.ColBERT(index=index)

In [17]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = models.ColBERT(
    model_name_or_path="lightonai/colbertv2.0",
    device=device,
)

query_text = "energy efficient fan"

Using device: cpu


In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"

def expand_query(query):
    data = {
        "model": "llama3.2",
        "stream": False,
        "messages": [
            {
                "role": "system",
                "content": "You are a search engine optimizer. Provide 5 synonyms or related product terms for the user query. Output ONLY the terms separated by commas. No explanations."
            },
            {
                "role": "user",
                "content": f"Query: {query}"
            }
        ]
    }
    response = requests.post(OLLAMA_URL, json=data)
    response.raise_for_status()
    content = response.json()["message"]["content"]
    terms = [t.strip() for t in content.split(",")]
    return terms

expanded_terms = expand_query(query_text)[:EXPAND_N]
all_queries = [query_text] + expanded_terms
print(f"Original:  {query_text}")
print(f"Expanded:  {expanded_terms}")

query_embeddings = model.encode(
    all_queries,
    batch_size=len(all_queries),
    is_query=True,
    show_progress_bar=False,
)

In [ ]:
EXPAND_N = 2
ORIGINAL_WEIGHT = 2.0

raw_results = retriever.retrieve(
    queries_embeddings=query_embeddings,
    k=10,
)

# Merge: keep best weighted score per document across all queries
# Original query gets ORIGINAL_WEIGHT boost; expanded queries get weight 1.0
merged = {}
for q_idx, query_results in enumerate(raw_results):
    weight = ORIGINAL_WEIGHT if q_idx == 0 else 1.0
    for hit in query_results:
        pid = hit["id"]
        score = hit["score"] * weight
        if pid not in merged or score > merged[pid]:
            merged[pid] = score

results = sorted(merged.items(), key=lambda x: x[1], reverse=True)[:5]
print(results)

In [20]:
import sqlite3

conn = sqlite3.connect("../data/processed/products_dataset.db")
cursor = conn.cursor()

print(f"Top results for query: '{query_text}' (with expansion)")
for product_id, score in results:
    cursor.execute("SELECT product_title FROM products WHERE pid = ?", (product_id,))
    row = cursor.fetchone()
    title = row[0] if row else "Unknown"
    print(f"[{score:.2f}] {product_id} - {title}")

conn.close()

Top results for query: 'energy efficient fan' (with expansion)
[23.47] 6 - Panasonic FV-08VRE2 Ventilation Fan with Recessed LED (Renewed)
[22.91] 46688 - JDSUMS USB Mini Fan,Handsfree Neck Fan Portable 5 Inch Hanging Fan, Powered by Rechargeable Battery with 3-Level Speed,Necklace Fan Multi-Functional for Travel,Sports, Office and Outdoor (White)
[22.53] 461 - 52 Inch Modern Ceiling Fan with Light Ceiling Fan Chandelier with Remote Control 3 Speed Reverse Function without Light Source E26 Suitable for Living Room, Bedroom, Dining Room
[22.44] 1 - Aero Pure AP80RVLW Super Quiet 80 CFM Recessed Fan/Light Bathroom Ventilation Fan with White Trim Ring
[22.07] 13640 - Nocolliny Modern Ceiling Fan 52" with Light, Dimmable LED, Farmhouse Style Propeller 3-Blade with Reversible Motor, Remote Control


In [21]:
# Clean up extracted index to save disk space
if index_dir.exists():
    shutil.rmtree(index_dir)
    print(f"Deleted extracted index: {index_dir}")

Deleted extracted index: ../data/processed/pylate_index
